# Spike Linear Model Comparison

Compare sigmoid (nonlinear) global epistasis to linear (Identity) models.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import multidms

from notebooks._common import load_config, all_muts_known, build_fit_params


In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
fit = spike["fitting"]

output_dir = spike["output_dir"]
reference = spike["reference"]
condition_colors = spike["condition_colors"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
train_frac = cfg["train_frac"]
seed = cfg["seed"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))

subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=seed)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
print(f"Loaded {len(func_score_df)} variants")

## Create Data objects and fit linear models

In [ ]:
cc = condition_colors
datasets = []
for res, fsdf in func_score_df.groupby("replicate"):
    df_agg = (
        fsdf.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")
    data = multidms.Data(
        df_agg,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=reference,
        assert_site_integrity=False,
        verbose=False,
        name=f"rep-{res}",
    )
    data.condition_colors = cc
    datasets.append(data)

fit_params_linear = build_fit_params(fit, datasets)
fit_params_linear["ge_type"] = ["Identity"]

_, _, linear_models = multidms.fit_models(
    fit_params_linear, gpu_ids=gpu_ids, n_processes=n_processes
)
for col in linear_models.columns:
    if linear_models[col].apply(lambda x: isinstance(x, dict)).any():
        linear_models[col] = linear_models[col].apply(str)

model_collection_linear = multidms.ModelCollection(linear_models)
print(f"Fit {len(linear_models)} linear models")

## Cross-validation for linear models

In [ ]:
# Build train/test split (same as spike_05)
train, test = [], {}
for replicate, fs_df in func_score_df.groupby("replicate"):
    df_agg = (
        fs_df.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")
    wt_mask = df_agg["aa_substitutions"].str.strip() == ""
    wt_rows = df_agg[wt_mask]
    non_wt = df_agg[~wt_mask].sample(frac=1, random_state=seed)
    n_train = int(len(non_wt) * train_frac)
    train_split = pd.concat([wt_rows, non_wt.iloc[:n_train]])
    test_split = non_wt.iloc[n_train:]
    name = f"rep-{replicate}"
    data = multidms.Data(
        train_split,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=reference,
        assert_site_integrity=False,
        verbose=False,
        name=name,
    )
    data.condition_colors = cc
    train.append(data)
    test[name] = test_split

fit_params_linear["dataset"] = train
_, _, linear_models_cv = multidms.model_collection.fit_models(
    fit_params_linear, gpu_ids=gpu_ids, n_processes=n_processes
)
for col in linear_models_cv.columns:
    if linear_models_cv[col].apply(lambda x: isinstance(x, dict)).any():
        linear_models_cv[col] = linear_models_cv[col].apply(str)

linear_mc = multidms.model_collection.ModelCollection(linear_models_cv)

filtered_test_linear = {}
for name, test_df in test.items():
    model = linear_mc.fit_models.query(f"dataset_name == '{name}'").iloc[0].model
    known_muts = set(model.data.mutations)
    mask = test_df["aa_substitutions"].apply(lambda s: all_muts_known(s, known_muts))
    filtered_test_linear[name] = test_df[mask]

linear_mc.add_eval_loss(filtered_test_linear, overwrite=True)
print(f"Fit {len(linear_models_cv)} linear CV models")

## Shrinkage analysis: linear models

In [ ]:
chart, sparsity_df = model_collection_linear.shift_sparsity(return_data=True, height_scalar=100)
chart, corr_df = model_collection_linear.mut_param_dataset_correlation(width_scalar=200, return_data=True)
cross_validation_df = linear_mc.loss_df()

saveas = "shrinkage_analysis_linear_models"
fig, ax = plt.subplots(3, figsize=[4.5, 7.5], sharex=True)

# Replicate correlation
iter_ax = ax[0]
sns.lineplot(
    data=(
        corr_df.query("mut_param.str.contains('shift')")
        .rename({"mut_param": "shift params"}, axis=1)
        .replace({"shift_Delta": "Delta", "shift_Omicron_BA2": "BA.2"})
        .assign(
            fusionreg=[f"{l:.1e}" for l in corr_df.query("mut_param.str.contains('shift')").fusionreg],
            correlation=lambda x: x.correlation ** 2,
        )
        .reset_index(drop=True)
    ),
    x="fusionreg", y="correlation", style="shift params",
    markers=True, ax=iter_ax, linewidth=3, color="black",
)
iter_ax.set_ylabel("rep1 v. rep2\nshift $(R^2)$")
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)

# Loss
iter_ax = ax[1]
sns.lineplot(
    data=cross_validation_df.query("condition=='total'").assign(
        lasso_strength=lambda x: x["fusionreg"].apply(lambda y: f"{y:.1e}")
    ),
    x="lasso_strength", y="loss", ax=iter_ax,
    hue="split", style="dataset_name",
    palette={"training": "slategrey", "validation": "#2CA02C"},
    markers=True, linewidth=3,
)
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)

# Sparsity
iter_ax = ax[2]
sns.lineplot(
    data=sparsity_df.rename({"dataset_name": "replicate", "mut_param": "shift params", "mut_type": "mutation type"}, axis=1)
    .replace({"shift_Delta": "Delta", "shift_Omicron_BA2": "BA.2"})
    .assign(
        fusionreg=[f"{l:.1e}" for l in sparsity_df.fusionreg],
        sparsity_percent=lambda x: x.sparsity * 100,
    ),
    x="fusionreg", y="sparsity_percent",
    hue="mutation type", style="replicate",
    palette={"nonsynonymous": "grey", "stop": "#E377C2"},
    markers=True, legend=True, ax=iter_ax, linewidth=3,
)
iter_ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)
iter_ax.set_xticklabels(iter_ax.get_xticklabels(), rotation=90, ha="center")
iter_ax.set_ylabel("sparsity\n$(\%\Delta=0)$")
iter_ax.set_xlabel("lasso regularization strength ($\lambda$)")

for axes in ax:
    axes.axvline(f"{chosen_lasso_strength:.1e}", color="grey", linewidth=10, alpha=0.35)

sns.despine(fig)
plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()